In [1]:
import numpy as np
import qtensor.states as states 
import qtensor.operators as ops
import qtensor.thermofield as thf
import qtensor.simulation.finiteTDVP as sim
import qtensor.simulation.updatemethod as methods
import copy

J=100
g=1
h=10

N=10
mid = N//2

H = ops.tilted_ising(J, g, h, N=10)

H_mid = ops.extensive_twosite_local_term(H, mid)


In [2]:
print([np.shape(H_mid.tensors[site]) for site in H_mid.tensors])

H_squared = H_mid @ H_mid

print([np.shape(H_squared.tensors[site]) for site in H_squared.tensors])

print(H_squared.l)

[(2, 2, 3, 3), (2, 2, 3, 3)]
[(2, 2, 9, 9), (2, 2, 9, 9)]
[1. 0. 0. 0. 0. 0. 0. 0. 0.]


In [3]:
# check that the norm local term is correct

print("Traces of squared local term:", H_squared.trace(N) )
normalized_trace = H_squared.trace(N) / (H.d ** N)
print("Normalized trace of squared local term:", normalized_trace)

Traces of squared local term: (10291712+0j)
Normalized trace of squared local term: (10050.5+0j)


In [4]:
H_1 = ops.extensive_twosite_local_term(H, mid)
H_2 = ops.extensive_twosite_local_term(H, mid+1)
H_3 = ops.extensive_twosite_local_term(H, mid+2)
H_terms = ops.extensive_as_terms(H)
print("Traces of H_1H_2:", (H_1 @ H_2).trace())
print("Traces of H_1H_3:", (H_1 @ H_3).trace())
trace_by_term = [(H_1 @ H_terms[site]).trace(N) for site in H_terms]
print("Traces of H_1H_terms:", sum(trace_by_term))
normalisation = (H.d ** N)
print("Normalised trace of H_iH:", sum(trace_by_term) / normalisation)
full_trace = (H_1 @ H).trace(N)
print("Full normed trace of H_1H:", full_trace / normalisation)

Traces of H_1H_2: (202+0j)
Traces of H_1H_3: 0j
Traces of H_1H_terms: (10343424+0j)
Normalised trace of H_iH: (10101+0j)
Full normed trace of H_1H: (10101+0j)


In [5]:
# Check commutator of paulis
x = ops.single_site_pauli(0, "x")
y = ops.single_site_pauli(0, "y")
com_xy = x @ y - y @ x
com_xy_squared = com_xy @ com_xy
print(com_xy[0][:, :, 0, 0])
print(com_xy.trace())
print(com_xy_squared.trace())
x_squared = x @ x
print("Normed trace of x^2:", x_squared.trace(N) / 2**N)

[[0.+1.j 0.+0.j]
 [0.+0.j 0.-1.j]]
0j
(-8+0j)
Normed trace of x^2: (1+0j)


In [3]:
J=100
h=10
g=1

N=10
mid = N//2

H = ops.tilted_ising(J, h, g, N=10)

H_mid = ops.extensive_twosite_local_term(H, mid)

H_1 = ops.extensive_twosite_local_term(H, mid)
H_2 = ops.extensive_twosite_local_term(H, mid+1)

# We can calculate the commutator trace in two ways,
# either by calculating the full commutator and then tracing,
# or by calculating the traces of the individual terms and then combining. 
# These should give the same result.
# For TFI my calculation says Tr([H_1, H_2]^2) = -2 * J^2 g^2 * 2^N
print("Prediction:", -2 * J**2 * g**2)

# Term by term trace
# [H1, H2]^2 = H1H2H1H2 - H1H2H2H1 - H2H1H1H2 + H2H1H2H1
# Tr([H1, H2]^2) = 2 * (Tr(H1H2H1H2) - Tr(H1H1H2H2)) by cyclicity
t1212 = (H_1@H_2@H_1@H_2).trace(N) / (2**N)
t1122 = (H_1@H_1@H_2@H_2).trace(N) / (2**N)
print("Term-by-termcommutator trace:", 2*(t1212 - t1122))

# Full commutator trace
com_12 = H_1 @ H_2 - H_2 @ H_1
com_12_squared = com_12 @ com_12
full_com_trace = com_12_squared.trace(N) / (2**N)
print("Full commutator trace:", full_com_trace)



Prediction: -20000
Term-by-termcommutator trace: (-20000+0j)
Full commutator trace: (-20000+0j)


In [1]:
# Both methods work. Lets try to calculate the more complicated 5-site one
 
import qtensor.operators as ops
import numpy as np

J=100
h=10
g=1

N=10
mid = N//2

H = ops.tilted_ising(J, h, g, N=N)
H_terms = ops.extensive_as_terms(H)

H_mid = ops.extensive_twosite_local_term(H, mid)

H_1 = ops.extensive_twosite_local_term(H, 1)
H_2 = ops.extensive_twosite_local_term(H, 2)
H_3 = ops.extensive_twosite_local_term(H, 3)
H_4 = ops.extensive_twosite_local_term(H, 4)

# Tr([H1, H2][H1, H2]) should be non-zero and
com_12 = H_1 @ H_2 - H_2 @ H_1
com_12_squared = com_12 @ com_12
full_com_trace = com_12_squared.trace(N) / (2**N)
print("Tr([H1, H2][H1, H2]):", full_com_trace)


# Tr([H1, H2][H1, H2]) should be zero for our model.
com_12 = H_1 @ H_2 - H_2 @ H_1
com_23 = H_2 @ H_3 - H_3 @ H_2
product = com_12 @ com_23
product_trace = product.trace(N) / (2**N)
print("Tr([H1, H2][H2, H3]):", product_trace)

# Tr([H_1, H_2][H_4, H_3]) should be zero since H_1 can commute around the trace.
com1 = H_1 @ H_2 - H_2 @ H_1
com2 = H_3 @ H_4 - H_4 @ H_3
com_product = com1 @ com2
com_product_trace = com_product.trace(N) / (2**N)
print("Tr([H1, H2][H3, H4]):", com_product_trace)

#should be the same as Tr([H_1, H][H_4, H])
com1_full = H_1 @ H - H @ H_1
com2_full = H_4 @ H - H @ H_4
com_product_full = com1_full @ com2_full
com_product_full_trace = com_product_full.trace(N) / (2**N)
print("Tr([H1, H][H4, H]):", com_product_full_trace)

# the above, but term by term
traces=0
for site in H_terms:
    H_i = H_terms[site]
    com1_full = H_1 @ H_i - H_i @ H_1
    com2_full = H_4 @ H_i - H_i @ H_4
    com_product_full = com1_full @ com2_full
    com_product_full_trace = com_product_full.trace(N) / (2**N)
    traces += com_product_full_trace
print("Tr([H1, H][H4, H]):", com_product_full_trace)

Tr([H1, H2][H1, H2]): (-20000+0j)
Tr([H1, H2][H2, H3]): 0j
Tr([H1, H2][H3, H4]): 0j
Tr([H1, H][H4, H]): (818241608+0j)
Tr([H1, H][H4, H]): 0j


In [4]:
for site in H_terms:
    print(f"Site {site}")
    H_i = H_terms[site]
    t00 = (H_i @ H @ H @ H_mid @ H).trace(N) / (2**N)
    t10 = (H @ H @ H_i @ H_mid @ H).trace(N) / (2**N)
    t01 = (H_i @ H @ H @ H @ H_mid).trace(N) / (2**N)
    t11 = (H @ H @ H_i @ H @ H_mid).trace(N) / (2**N)
    print("Tr([H_i, H H] [H_mid, H]):", t00 - t01 -t10 + t11)


# Going term by term with the commutators

Site 0
Tr([H_i, H H] [H_mid, H]): 0j
Site 1
Tr([H_i, H H] [H_mid, H]): 0j
Site 2
Tr([H_i, H H] [H_mid, H]): 0j
Site 3
Tr([H_i, H H] [H_mid, H]): 0j
Site 4
Tr([H_i, H H] [H_mid, H]): 0j
Site 5
Tr([H_i, H H] [H_mid, H]): 0j
Site 6
Tr([H_i, H H] [H_mid, H]): 0j
Site 7
Tr([H_i, H H] [H_mid, H]): 0j
Site 8
Tr([H_i, H H] [H_mid, H]): 0j
